# Wavelet-PINN for the 2D elliptic interface problem (circular interface)
Decomposed wavelet expansions (one per subdomain) coupled by the interface conditions, solved AD-free by weighted least squares.
$-\nabla\cdot(a\nabla u)=f$, $a=a_{in}$ inside / $a_{out}$ outside; $[[u]]=0$, $[[a\,\partial_n u]]=0$ on $\Gamma$.

In [ ]:
from config import *
from EllipticInterface import *
from Wfamily import *
from Model import *
import math

In [ ]:
# Assemble the decomposed least-squares system for theta = [c^-, c^+, b^-, b^+]
NF = len_family; n = 2*NF + 2
wi, wb, tik = 10.0, 10.0, 1e-10   # interface / boundary weights, Tikhonov

def blk(rows, cM=None, cP=None, bM=None, bP=None):
    B = torch.zeros(rows, n)
    if cM is not None: B[:, :NF] = cM
    if cP is not None: B[:, NF:2*NF] = cP
    if bM is not None: B[:, 2*NF] = bM
    if bP is not None: B[:, 2*NF+1] = bP
    return B

A = torch.cat([
    blk(len(x_in),  cM=-a_in*Lin),                              # interior residual Omega^-
    blk(len(x_out), cP=-a_out*Lout),                            # interior residual Omega^+
    math.sqrt(wi)*blk(len(x_if), cP=Wif,  cM=-Wif, bP=1., bM=-1.),   # [[u]] = 0
    math.sqrt(wi)*blk(len(x_if), cP=a_out*DnIf, cM=-a_in*DnIf),      # [[a d_n u]] = 0
    math.sqrt(wb)*blk(len(x_bc), cP=Wbc, bP=1.),               # Dirichlet BC (Omega^+)
], 0)
b = torch.cat([
    torch.full((len(x_in),),  fval), torch.full((len(x_out),), fval),
    torch.zeros(len(x_if)), torch.zeros(len(x_if)),
    math.sqrt(wb)*torch.tensor(u_bc),
]).unsqueeze(1)
print('system:', tuple(A.shape))

In [ ]:
# Column-normalised (Jacobi-preconditioned) + Tikhonov least-squares solve -- AD-free, ~sub-second
s = A.norm(dim=0).clamp_min(1e-30)
Aa = torch.cat([A/s, math.sqrt(tik)*torch.eye(n)])
bb = torch.cat([b, torch.zeros(n, 1)])
theta = (torch.linalg.lstsq(Aa, bb, driver='gelsd').solution.squeeze(1)) / s
cM, cP, bM, bP = theta[:NF], theta[NF:2*NF], theta[2*NF], theta[2*NF+1]

In [ ]:
# Evaluate, report error, and plot sol.png
um = (Wtest@cM + bM).numpy(); up = (Wtest@cP + bP).numpy()
rr = x_test**2 + y_test**2
pred = np.where(rr < r0*r0, um, up)
relL2 = np.linalg.norm(pred - u_exact_te)/np.linalg.norm(u_exact_te)
jump_pred = float(((DnIf@cP) - (DnIf@cM)).mean().item())
print(f'rel L2 = {relL2:.3e} | kink pred = {jump_pred:.4f} (exact {jump_dn:.4f})')

PR = pred.reshape(Xt.shape); UE = u_exact_te.reshape(Xt.shape); ER = np.abs(PR-UE)
th = np.linspace(0, 2*np.pi, 200)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for a_, Z, ttl, cm in [(ax[0],UE,'exact','viridis'),(ax[1],PR,'wavelet W-PINN','viridis'),
                       (ax[2],np.log10(ER+1e-16),'log10 |error|','magma')]:
    c = a_.contourf(Xt, Yt, Z, 40, cmap=cm); a_.plot(r0*np.cos(th), r0*np.sin(th), 'w--', lw=1)
    a_.set_aspect('equal'); a_.set_title(ttl); fig.colorbar(c, ax=a_)
plt.tight_layout(); plt.savefig('sol.png', dpi=120); plt.show()